# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, validate in time order, compare baseline / logistic / gradient boosting, report honestly.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'missing:', int(df.isna().sum().sum()), 'dupes:', int(df.duplicated().sum()))
print('unique assets:', df.asset_tag.nunique(), '| range:', df.inspected_at.min(), '->', df.inspected_at.max())
print('base rate:', df.failed_90d.mean().round(4))
df.describe().T

## Leakage check
`parts_replaced` is the number of parts replaced on the work order that *closed* the inspection record, so it may be filled in after a failure.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

**Finding:** 0 parts -> never fails; 2 or more parts -> always fails; 1 part -> about 27%. Near-perfect separation is not plausible as a prior risk signal; it is almost certainly a consequence of the failure/repair and would **not be available at inspection time**. I exclude it and show below how much it inflates scores. `asset_tag` is unique per row (a pure ID), so it is dropped too.

## Exploration

In [ ]:
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 5, figsize=(18, 3))
for a, col in zip(ax, num):
    for v, l in [(0, 'ok'), (1, 'failed')]:
        a.hist(df.loc[df.failed_90d == v, col], bins=30, alpha=.55, density=True, label=l)
    a.set_title(col)
ax[0].legend(); plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.agg(['mean', 'size']))
print(df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'size']))

The failure rate swings strongly by quarter. Part of that may be right-censoring at the end of the data, but it also drifts earlier, so a random split would be optimistic. I use forward-chaining time splits (train on the past, test on the future).

## Modelling (time-ordered CV, `parts_replaced` excluded)

In [ ]:
df['bearing_minus_ambient'] = df.bearing_temp_c - df.ambient_temp_c
nums = num + ['bearing_minus_ambient']; feats = nums + ['site']
y = df.failed_90d.values
pre = ColumnTransformer([('n', StandardScaler(), nums), ('c', OneHotEncoder(), ['site'])])
models = {
    'baseline (prior)': DummyClassifier(strategy='prior'),
    'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
    'hist-GBM': make_pipeline(ColumnTransformer([('c', OneHotEncoder(), ['site'])], remainder='passthrough'),
                              HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, l2_regularization=1.0, random_state=0)),
}
tscv = TimeSeriesSplit(n_splits=5)
def evaluate(X, models):
    rows = []
    for name, m in models.items():
        a = []; p = []; b = []
        for tr, te in tscv.split(X):
            m.fit(X.iloc[tr], y[tr]); pr = m.predict_proba(X.iloc[te])[:, 1]
            a.append(roc_auc_score(y[te], pr)); p.append(average_precision_score(y[te], pr)); b.append(brier_score_loss(y[te], pr))
        rows.append((name, np.mean(a), np.std(a), np.mean(p), np.mean(b)))
    return pd.DataFrame(rows, columns=['model', 'ROC AUC', 'AUC sd', 'PR AUC', 'Brier']).round(3)
X = df[feats]
res = evaluate(X, models); print(res.to_string(index=False))
print('test-fold prevalence:', [round(y[te].mean(), 3) for _, te in tscv.split(X)])

### What if `parts_replaced` were (wrongly) included?

In [ ]:
Xl = df[feats + ['parts_replaced']]
pre_l = ColumnTransformer([('n', StandardScaler(), nums + ['parts_replaced']), ('c', OneHotEncoder(), ['site'])])
leak = {'logistic + parts_replaced': make_pipeline(pre_l, LogisticRegression(max_iter=1000)),
        'hist-GBM + parts_replaced': make_pipeline(ColumnTransformer([('c', OneHotEncoder(), ['site'])], remainder='passthrough'),
                                                   HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0))}
print(evaluate(Xl, leak).to_string(index=False))

## Feature importance & calibration (last fold)

In [ ]:
tr, te = list(tscv.split(X))[-1]
m = models['hist-GBM'].fit(X.iloc[tr], y[tr])
pi = permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=20, random_state=0)
print(pd.Series(pi.importances_mean, index=feats).sort_values().round(4))
pr = m.predict_proba(X.iloc[te])[:, 1]
print(pd.DataFrame({'pred': pr, 'obs': y[te]}).groupby(pd.qcut(pr, 5, duplicates='drop'), observed=True).mean().round(3))
lr = models['logistic'].fit(X.iloc[tr], y[tr])
print(pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out()).round(3))

## Is the signal stable over time?
Single-feature ROC AUC per quarter (0.5 = no signal; below 0.5 = reversed direction).

In [ ]:
d = df.assign(q=df.inspected_at.dt.to_period('Q'))
pd.DataFrame({q: {c: roc_auc_score(g.failed_90d, g[c]) for c in num} for q, g in d.groupby('q')}).T.round(3)

## Conclusion
(Numbers below are from a run of this notebook; re-run to refresh the cell outputs.)

**Yes, partially, and only once a leaky column is excluded.**

- **`parts_replaced` must not be used.** With it, AUC is about 0.998 (0 parts -> never fails, 2+ parts -> always fails). It looks like a post-failure work-order artefact. Please confirm with the owner of the work-order system when it is filled in. If it really is known at inspection time, the question changes.
- **Honest models** (time-ordered 5-fold CV): baseline AUC 0.50; logistic 0.70 +/- 0.05; gradient boosting 0.76 +/- 0.07, PR AUC about 0.28 against fold prevalence of roughly 2-35%. That is better than chance and the GBM ranks risk reasonably well in the last fold (top quintile predicted 16%, observed 9%; bottom quintiles near 0%), but it is a modest, noisy signal, not a reliable per-asset alarm.
- **Drivers:** high load, high vibration and high bearing temperature go with more failures. Site is irrelevant.
- **Instability:** the failure rate swings from 4% to 27% to 3% across quarters, and single-feature signals drift (vibration AUC falls from about 0.78 in 2024Q1 to about 0.43 in 2025Q4, while load rises from about 0.60 to 0.82; ambient temp and meter hours flip direction). Expect performance to degrade without monitoring and retraining.
- **Caveats:** one row per asset, so no history features (prior inspections, trends), which are probably the best next step; late-2025 labels may be incomplete (right-censoring); no cost for false alarms vs misses, so no decision threshold is chosen.